# REALISTA — Demo: Attacking an Open-Source Model

This notebook runs REALISTA end-to-end on a **single MMLU example**, attacking an open-source
target model (e.g. Llama-3.1-8B-Instruct) directly — no external reasoning target.

REALISTA is a two-stage attack:

1. **Stage 1** scores a set of candidate concept-based rephrasings of the question (drawn from a
   provided rephrasing dictionary) and picks the best one per concept.
2. **Stage 2** runs Projected Langevin Dynamics (PLD): a sparse latent perturbation
   `delta` over a dynamic concept dictionary `D(Z0)` is optimized via Langevin dynamics so that
   decoding the perturbed latent yields a semantically-equivalent rephrasing that flips the
   target model's answer.

**This notebook assumes the two REALISTA dictionaries already exist** (see `config.py`):
- a rephrasing dictionary at `REPHRASING_DICT_PATH` (stage 1)
- a latent-direction dictionary at `LATENT_DICT_PATH` (stage 2)

This code only *loads and uses* those dictionaries — building them is out of scope here.

Run this notebook from inside the `src/` folder (or add it to `sys.path`) so the flat module
imports below resolve correctly.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
from datasets import load_dataset

from arguments import RealistaArgs
from config import MMLU_DATASET
from model_utils import load_model_and_tokenizer
from qa_utils import get_ground_truth_target_choice, print_full_prompt, format_probs
from dictionary_utils import get_Z0, load_latent_dict, get_dictionary_and_base_latent
from realista import stage1_optimization, PLD
from utils import set_seed


## 0. Check GPU availability


In [2]:
import torch

if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected -- running the target model on CPU will be extremely slow.")


CUDA available: NVIDIA RTX A5000


## 1. Configure the attack

Pick a target model (`config.MODEL_REGISTRY`), an MMLU subject, and a single question index.


In [3]:
args = RealistaArgs(
    model_type="llama3_8b",       # one of config.MODEL_REGISTRY
    mmlu_subject="machine_learning",
    mmlu_question_idx=17,
    trial_num=10,                  # number of stage-2 PLD trials
    max_iter=10,                   # number of PLD steps per trial
    epsilon=1.0,
    eta=1e-3,
    T0=1.0,
    num_concepts=None,             # None = use every concept in the rephrasing dictionary
    num_rephrasings_per_concept=None,
    verbose=False,                 # True = full per-step text dump; False = compact progress table
)
set_seed(args.seed)
args


RealistaArgs(mmlu_subject='machine_learning', mmlu_question_idx=17, seed=18, model_type='llama3_8b', trial_num=10, reasoning_target='none', stage1_batch_size=16, num_concepts=None, num_rephrasings_per_concept=None, epsilon=1.0, eta=0.001, max_iter=10, T0=1.0, annealing_rate=0.9, prompt_len=50, noise_only=False, gradient_only=False, verbose=False)

## 2. Load the target model

In [4]:
model, tokenizer = load_model_and_tokenizer(args.model_type)


Loading target LLM: llama3_8b (meta-llama/Llama-3.1-8B-Instruct)


2026-07-06 18:48:38.649876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783378118.661170  844734 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783378118.664165  844734 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783378118.672545  844734 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783378118.672553  844734 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783378118.672555  844734 computation_placer.cc:177] computation placer alr

## 3. Load one MMLU example

In [5]:
mmlu_dataset = load_dataset(MMLU_DATASET, args.mmlu_subject)
cur_task_dict = mmlu_dataset["test"][args.mmlu_question_idx]
print_full_prompt(cur_task_dict)


Full Input Prompt:
You are the world's best expert in answering questions related to machine learning. Answer the following question and give me the reason. 
For a neural network, which one of these structural assumptions is the one that most affects the trade-off between underfitting (i.e. a high bias model) and overfitting (i.e. a high variance model):
    A. The number of hidden nodes
    B. The learning rate
    C. The initial choice of weights
    D. The use of a constant-term unit input
The correct answer is option: 


## 4. Base latent `Z0` and the attack target

`target_choice_index` is the answer choice REALISTA will try to make the model prefer instead
of the ground truth (the model's second-most-confident choice, or its current top choice if
it is already wrong).


In [6]:
Z0_unpadded = get_Z0(cur_task_dict, args, model, tokenizer)
ground_truth_idx, target_choice_index = get_ground_truth_target_choice(
    args, cur_task_dict, model, tokenizer, Z0_unpadded
)
print(f"Ground truth choice index: {ground_truth_idx}")
print(f"Target (adversarial) choice index: {target_choice_index}")


Ground Truth Index: 0  |  Target Choice Index: 3
Ground truth choice index: 0
Target (adversarial) choice index: 3


## 5. Stage 1 — select the best candidate rephrasing per concept

Loads the provided rephrasing dictionary for this subject/question and scores every candidate
rephrasing under the target model.


In [7]:
stage1_result = stage1_optimization(args, model, tokenizer, cur_task_dict, target_choice_index)

best_concept_key = max(stage1_result, key=lambda k: stage1_result[k]["obj_value"])
print("Best stage-1 candidate:", best_concept_key)
print(stage1_result[best_concept_key])



Stage 1: Single-Concept Initialization


Stage 1 batches: 100%|██████████| 19/19 [00:08<00:00,  2.25it/s]

(* marks the ground-truth answer; ^ marks the target choice REALISTA optimizes toward)
Concept                                    Objective        A*         B         C        D^
--------------------------------------------------------------------------------------------
trial-and-error_0                            -1.3848    55.56%     9.28%    10.12%    25.05%
trial-and-error_1                            -1.8672    73.75%     7.18%     3.64%    15.45%
trial-and-error_2                            -2.0586    78.19%     4.62%     4.45%    12.76%
metonymic_0                                  -1.6309    68.81%     4.54%     7.09%    19.56%
metonymic_1                                  -1.8389    78.81%     2.94%     2.34%    15.89%
metonymic_2                                  -2.0156    76.69%     5.96%     4.03%    13.33%
at fault_0                                   -1.3867    56.34%    10.19%     8.44%    25.00%
at fault_1                                   -1.3438    56.06%     9.08%    

## 6. Stage 2 — PLD over the latent concept dictionary

Loads the provided latent-direction dictionary, builds the dynamic concept dictionary `D(Z0)`,
and runs PLD initialized from the stage-1 result.


In [8]:
latent_dict = load_latent_dict(args.model_type, args.mmlu_subject, verbose=args.verbose)
D_Z0, Z0, cur_task_dict, concepts = get_dictionary_and_base_latent(args, model, tokenizer, latent_dict)

result_dict_ls = PLD(
    args, model, tokenizer, Z0, D_Z0, concepts, cur_task_dict, target_choice_index, stage1_result
)


Padding perturbation directions: 100%|██████████| 100/100 [00:00<00:00, 805.46it/s]



Stage 2: Refinement with Stochastic Exploration
(* marks the ground-truth answer; ^ marks the target choice REALISTA optimizes toward)

Trial 1/10
 Step   Objective        A*         B         C        D^  Active     Temp  Feasible
------------------------------------------------------------------------------------
    0     -1.5938    56.94%    14.98%     7.77%    20.31%       0   1.0000         -
    1     -0.5020    22.98%     6.64%     9.80%    60.56%       1   1.0000        No
    2     -0.5020    22.98%     6.64%     9.80%    60.56%      10   0.9000       Yes
    3     -0.5020    22.98%     6.64%     9.80%    60.56%      10   0.8100         -
    4     -0.5020    22.98%     6.64%     9.80%    60.56%      10   0.7290         -
    5     -1.2412    47.28%     9.91%    13.87%    28.91%      10   0.6561         -
    6     -1.5889    68.56%     7.11%     3.93%    20.42%      10   0.5905         -
    7     -1.5195    65.88%     7.80%     4.45%    21.89%      27   0.5314         -
  

## 7. Final adversarial prompt

In [9]:
best_trial = max(result_dict_ls, key=lambda r: r["obj_best"])

print("Answer Options:")
for i, (letter, choice) in enumerate(zip(["A", "B", "C", "D"], cur_task_dict["choices"])):
    if i == best_trial["ground_truth_idx"]:
        tag = "  <-- GROUND TRUTH"
    elif i == target_choice_index:
        tag = "  <-- TARGET (REALISTA's goal)"
    else:
        tag = ""
    print(f"  {letter}. {choice}{tag}")

print(f"\nOriginal Objective Value: {best_trial['obj_values'][0]:.4f}")
print(f"Original Answer Probabilities: {format_probs(best_trial['probs'][0])}")

print(f"\nBest Objective Value: {best_trial['obj_best']:.4f}  (Trial {best_trial['trial_idx'] + 1}, Iteration {best_trial['iter_best']})")
print(f"Answer Probabilities: {format_probs(best_trial['probs'][best_trial['iter_best']])}")

print(f"\nOriginal Prompt: {cur_task_dict['question']}")
print(f"Original Answer: {best_trial['vanilla_response']}")
print(f"\nAdversarial Prompt: {best_trial['attack_prompts'][best_trial['iter_best']]}")
print(f"Adversarial Answer: {best_trial['best_response']}")


Answer Options:
  A. The number of hidden nodes  <-- GROUND TRUTH
  B. The learning rate
  C. The initial choice of weights
  D. The use of a constant-term unit input  <-- TARGET (REALISTA's goal)

Original Objective Value: -1.5938
Original Answer Probabilities: A: 56.93%  B: 14.98%  C:  7.77%  D: 20.31%

Best Objective Value: -0.5020  (Trial 1, Iteration 2)
Answer Probabilities: A: 22.99%  B:  6.64%  C:  9.81%  D: 60.55%

Original Prompt: For a neural network, which one of these structural assumptions is the one that most affects the trade-off between underfitting (i.e. a high bias model) and overfitting (i.e. a high variance model):
Original Answer:  A. The number of hidden nodes
The reason is: The number of hidden nodes in a neural network determines the capacity of the model to fit the training data. If the number of hidden nodes is too small, the model may not be able to capture the underlying patterns in the data, leading to underfitting. On the other hand, if the number of hidde